# PEPITA on MobileNetV1 / Tiny ImageNet: results from the stored weights

This notebook scores the stored weight files on the noisy Tiny ImageNet test set and writes the results to an Excel
workbook. It trains nothing and reads no other notebook, script or results file: everything it needs is in the cells
below.

| Row | Weight file, for input-noise variance *v* |
|---|---|
| Init (no retrain) | `weights_baselines/full_retrain_var_0.0.pth`, the clean model every run starts from |
| PEPITA | `weights_pepita_mobilenetv1_tinyimagenet/pepita_var_<v>.pth`, a MobileNetV1 state dict (PEPITA adapts a plain MobileNetV1) |

- **Test set:** the 10,000 official Tiny ImageNet validation images, in `val/val_annotations.txt` order, labelled with
  the class index of their wnid among the sorted `train/` folder names. Gaussian noise of variance *v* is added after
  normalisation, drawn on the CPU with seed 42 at every level.
- **Model:** MobileNetV1 (width 1.0, stride-1 stem, 200 classes) in eval mode, so BatchNorm uses the running statistics
  stored in each file. No test-time augmentation, as in the PEPITA sweep; non-finite logits, if any, are set to 0.
- **Metrics:** accuracy, macro precision, recall and F1 over the 200 classes, and mAP (the mean one-vs-rest average
  precision on the softmax scores), rounded to 6 decimals. Accuracies can differ by a few test images between machines
  (GPU and CPU arithmetic).
- **Output:**
  `mobilenetv1_tinyimagenet/results_pepita_mobilenetv1_tinyimagenet/results_pepita_mobilenetv1_tinyimagenet_reproduced.xlsx`,
  one sheet `long` with a row per noise level and method; a re-run overwrites it. The name carries no `YYYYmmdd_HHMMSS`
  stamp, so the plotting notebooks, which read the workbook with the newest stamp in each results folder, keep using the
  reported one.

**How to run:** run all cells. The notebook looks for the project folder (the one holding `mobilenetv1_tinyimagenet/`
and `data/`) from its own folder upwards and reads Tiny ImageNet from `data/tiny-imagenet-200` (unzip
`tiny-imagenet-200.zip` there if it is missing). It needs torch, torchvision, numpy, pandas, Pillow and openpyxl.

On a CPU this takes a while: 20 evaluations of 10,000 images, one pass each, about 25 minutes on a 6-core machine. A GPU
is much faster (set `DEVICE` below).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import datasets, transforms

## Settings

In [2]:
# ---- settings: edit here if needed ----
DATASET = "mobilenetv1_tinyimagenet"
METHOD_KEY = "pepita"                  # the weights folder is weights_<METHOD_KEY>_mobilenetv1_tinyimagenet/
METHOD_LABEL = "PEPITA"                # the method's row label in the workbook

DEVICE = "cpu"                         # "cuda" works too, on a GPU this torch build supports
BATCH_SIZE = 1024                      # evaluation batch; lower it if memory is short
SEED = 42
NOISE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]   # input-noise variances
TEST_NOISE_SEED = SEED                 # the noise stream of the PEPITA sweep
HFLIP_TTA = False                      # the PEPITA sweep scores without test-time augmentation
NUM_CLASSES = 200


def find_project_root(start=Path.cwd()):
    """The BOLSTER folder: the first folder, from the notebook's folder upwards, that holds DATASET/ and data/."""
    for c in (start, *start.parents):
        if (c / DATASET).is_dir() and (c / "data").is_dir():
            return c
    raise FileNotFoundError(f"no folder with {DATASET}/ and data/ above {start}")


PROJECT = find_project_root()
DATA_ROOT = PROJECT / "data"
BASELINE_WEIGHTS = PROJECT / DATASET / "weights_baselines"
METHOD_WEIGHTS = PROJECT / DATASET / f"weights_{METHOD_KEY}_{DATASET}"
RESULTS_DIR = PROJECT / DATASET / f"results_{METHOD_KEY}_{DATASET}"
# no YYYYmmdd_HHMMSS stamp in the name: the plotting notebooks keep reading the reported workbook
XLSX_PATH = RESULTS_DIR / f"results_{METHOD_KEY}_{DATASET}_reproduced.xlsx"

print(f"project: {PROJECT}\ndevice: {DEVICE}")

project: /home/rahman2/GIT/BOLSTER
device: cpu


## The network

In [3]:
class DWSeparable(nn.Module):
    """A 3x3 depthwise conv (+BN+ReLU), then a 1x1 pointwise conv (+BN+ReLU)."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw  = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch, bias=False)
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.pw  = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        x = F.relu(self.bn1(self.dw(x)))
        x = F.relu(self.bn2(self.pw(x)))
        return x


class MobileNetV1(nn.Module):
    """MobileNet-v1 (Howard et al. 2017), width 1.0, with a 3x3 stride-1 stem so that 64x64 Tiny ImageNet keeps its
    resolution (64 -> 32 -> 16 -> 8 -> 4 before global average pooling), 13 depthwise-separable blocks and a linear
    head. The module names match the stored state dicts."""
    # (out_channels, stride) per depthwise-separable block
    cfg = [(64, 1), (128, 2), (128, 1), (256, 2), (256, 1), (512, 2),
           (512, 1), (512, 1), (512, 1), (512, 1), (512, 1), (1024, 2), (1024, 1)]

    def __init__(self, num_classes=NUM_CLASSES, in_ch=3, stem_stride=1):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, stride=stem_stride, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        blocks, c = [], 32
        for out_ch, s in self.cfg:
            blocks.append(DWSeparable(c, out_ch, stride=s)); c = out_ch
        self.features = nn.Sequential(*blocks)
        self.fc = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.fc(x)

## Metrics

In [4]:
def macro_prf(y_true, y_pred, num_classes=NUM_CLASSES):
    """Macro precision, recall and F1 over the classes; a class with an empty denominator contributes 0."""
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    np.add.at(cm, (y_true, y_pred), 1)
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    rec  = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    den  = prec + rec
    f1   = np.divide(2 * prec * rec, den, out=np.zeros_like(tp), where=den > 0)
    return float(prec.mean()), float(rec.mean()), float(f1.mean())


def average_precision(is_pos, scores):
    """One-vs-rest average precision of one class (the scikit-learn definition: the sum over the distinct score
    thresholds of (R_n - R_{n-1}) P_n)."""
    n_pos = int(is_pos.sum())
    if n_pos == 0:
        return 0.0
    order = np.argsort(-scores, kind="mergesort")
    y, s = is_pos[order], scores[order]
    last = np.r_[np.where(np.diff(s))[0], y.size - 1]        # the last index of each distinct score
    tps = np.cumsum(y)[last]
    fps = (1 + last) - tps
    precision = np.r_[1.0, tps / (tps + fps)]
    recall    = np.r_[0.0, tps / n_pos]
    return float(np.sum(np.diff(recall) * precision[1:]))


def finite_logits(z):
    """Non-finite logits (a diverged model) set to 0, so that such a model scores like a uniform predictor."""
    return z if torch.isfinite(z).all() else torch.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)


@torch.no_grad()
def eval_metrics(state_dict, x, y, model_cls=MobileNetV1):
    """accuracy, macro precision / recall / F1 and mAP of the model in state_dict on (x, y). The argmax gives the
    predictions, the softmax the scores for mAP."""
    model = model_cls().to(DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    probs, preds = [], []
    for i in range(0, y.numel(), BATCH_SIZE):
        xb = x[i:i + BATCH_SIZE].to(DEVICE)
        logits = finite_logits(model(xb))
        if HFLIP_TTA:                          # the mean with the logits of the mirror images
            logits = 0.5 * (logits + finite_logits(model(xb.flip(-1))))
        probs.append(F.softmax(logits, dim=1).cpu())
        preds.append(logits.argmax(1).cpu())
    probs, y_pred, y_true = torch.cat(probs).numpy(), torch.cat(preds).numpy(), y.numpy()
    precision, recall, f1 = macro_prf(y_true, y_pred)
    mAP = float(np.mean([average_precision((y_true == c).astype(np.int64), probs[:, c]) for c in range(NUM_CLASSES)]))
    return {"accuracy": float((y_pred == y_true).mean()), "precision": precision, "recall": recall, "f1": f1,
            "map": mAP}

## The test set
The official validation images; the training images are not loaded.

In [5]:
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4802, 0.4481, 0.3975), (0.2770, 0.2691, 0.2821)),
])
TINY_ROOT = DATA_ROOT / "tiny-imagenet-200"


def add_noise(x, var, seed):
    """x plus Gaussian noise of variance var, drawn on the CPU from a generator seeded with seed."""
    if var <= 0:
        return x
    g = torch.Generator().manual_seed(seed)
    return x + torch.randn(x.shape, generator=g) * float(np.sqrt(var))


# the class index of a wnid: its position among the sorted train/ folder names (torchvision's ImageFolder order)
wnids = sorted(d.name for d in (TINY_ROOT / "train").iterdir() if d.is_dir())
class_to_idx = {w: i for i, w in enumerate(wnids)}

images, labels = [], []
with open(TINY_ROOT / "val" / "val_annotations.txt") as f:
    for line in f:                                   # <file name> <wnid> <bounding box>
        name, wnid = line.split("\t")[:2]
        images.append(tfm(Image.open(TINY_ROOT / "val" / "images" / name).convert("RGB")))
        labels.append(class_to_idx[wnid])
x_test, y_test = torch.stack(images), torch.tensor(labels, dtype=torch.long)
del images, labels
print(f"test set: the {y_test.numel():,} official validation images, {len(wnids)} classes")

test set: the 10,000 official validation images, 200 classes


## Scores and workbook
Which weight file feeds which row; every row is scored on that level's noisy test set, then the workbook is written.

In [6]:
ROWS = [                               # (row label, weight file for noise variance v), in the workbook's order
    ("Init (no retrain)", lambda v: BASELINE_WEIGHTS / "full_retrain_var_0.0.pth"),
    (METHOD_LABEL, lambda v: METHOD_WEIGHTS / f"{METHOD_KEY}_var_{v}.pth"),
]
METRIC_COLS = ["accuracy", "precision", "recall", "f1", "map"]


def load_state(path):
    return torch.load(path, map_location="cpu", weights_only=True)


records = []
for v in NOISE_LEVELS:
    level = f"{v:.0%}"
    x_noisy = add_noise(x_test, v, seed=TEST_NOISE_SEED)
    for label, path_of in ROWS:
        path = path_of(v)
        r = eval_metrics(load_state(path), x_noisy, y_test)
        records.append({"noise_level": level, "method": label, **{m: round(r[m], 6) for m in METRIC_COLS}})
        print(f"{level:>5}  {label:<20} acc {r['accuracy']:.6f}  f1 {r['f1']:.6f}  map {r['map']:.6f}   <- {path.name}")

results = pd.DataFrame(records, columns=["noise_level", "method", *METRIC_COLS])
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results.to_excel(XLSX_PATH, sheet_name="long", index=False)
print(f"\nwrote {XLSX_PATH.relative_to(PROJECT)}")

  10%  Init (no retrain)    acc 0.270000  f1 0.259888  map 0.327048   <- full_retrain_var_0.0.pth


  10%  PEPITA               acc 0.418600  f1 0.417478  map 0.415083   <- pepita_var_0.1.pth


  20%  Init (no retrain)    acc 0.142700  f1 0.133102  map 0.195945   <- full_retrain_var_0.0.pth


  20%  PEPITA               acc 0.358000  f1 0.357289  map 0.339681   <- pepita_var_0.2.pth


  30%  Init (no retrain)    acc 0.091900  f1 0.080207  map 0.132073   <- full_retrain_var_0.0.pth


  30%  PEPITA               acc 0.311800  f1 0.310823  map 0.287331   <- pepita_var_0.3.pth


... (10 outputs left out)

  90%  Init (no retrain)    acc 0.025700  f1 0.011701  map 0.035144   <- full_retrain_var_0.0.pth


  90%  PEPITA               acc 0.171200  f1 0.169868  map 0.140766   <- pepita_var_0.9.pth


 100%  Init (no retrain)    acc 0.024000  f1 0.010102  map 0.030983   <- full_retrain_var_0.0.pth


 100%  PEPITA               acc 0.161800  f1 0.159981  map 0.129330   <- pepita_var_1.0.pth

wrote mobilenetv1_tinyimagenet/results_pepita_mobilenetv1_tinyimagenet/results_pepita_mobilenetv1_tinyimagenet_reproduced.xlsx
